# PLEM — Metric Comparison Notebook

**Research question:** IoU is the standard evaluation metric for segmentation, but it is poorly suited to linear features (roads). This notebook demonstrates:

1. **Why IoU fails for roads** — sensitivity curves showing IoU collapses for small offsets and over-thick predictions where clDice/DTAF1 do not.
2. **Proposed unified metrics** — DTAF1 and CBHM evaluated on synthetic scenes.
3. **Sanity checks** — perfect and null predictions, class-isolation.
4. **Sensitivity sweeps** — road offset, road breakage, building erosion, road width.


In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import binary_erosion

from metrics.dtaf1 import dtaf1
from metrics.cldice import cldice
from metrics.boundary_f1 import boundary_f1, iou
from metrics.unified import cbhm, evaluate_all

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('All imports OK')

## 1. Synthetic Scene

We create a 128×128 label map with:
- **Class 1 (road):** vertical stripe through the centre
- **Class 2 (building):** 40×40 rectangle in the upper-left quadrant

In [ ]:
SHAPE = (128, 128)

def make_road(col=64, thickness=3, shape=SHAPE):
    m = np.zeros(shape, dtype=np.uint8)
    half = thickness // 2
    m[:, col - half: col + half + 1] = 1
    return m

def make_building(top=20, left=20, h=40, w=40, shape=SHAPE):
    m = np.zeros(shape, dtype=np.uint8)
    m[top: top+h, left: left+w] = 2
    return m

def make_gt(shape=SHAPE):
    gt = np.zeros(shape, dtype=np.uint8)
    gt[make_road(shape=shape) > 0] = 1
    gt[make_building(shape=shape) > 0] = 2
    return gt

GT = make_gt()

cmap = plt.cm.get_cmap('tab10', 3)
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(GT, cmap=cmap, vmin=0, vmax=2)
patches = [
    mpatches.Patch(color=cmap(0), label='Background (0)'),
    mpatches.Patch(color=cmap(1), label='Road (1)'),
    mpatches.Patch(color=cmap(2), label='Building (2)'),
]
ax.legend(handles=patches, loc='lower right', fontsize=9)
ax.set_title('Ground Truth label map')
plt.tight_layout()
plt.show()

## 2. Sanity Checks

In [ ]:
ROAD_CONFIG = {
    1: {'name': 'road',     'tolerance': 10},
    2: {'name': 'building', 'tolerance':  2},
}

# Perfect prediction
r_perfect = evaluate_all(GT, GT)
print('=== Perfect prediction (pred == GT) ===')
print(f"  DTAF1 = {r_perfect['dtaf1']:.4f}  (expected 1.0)")
print(f"  CBHM  = {r_perfect['cbhm']:.4f}  (expected 1.0)")

# Null prediction
NULL = np.zeros_like(GT)
r_null = evaluate_all(NULL, GT)
print('\n=== Null prediction (all zeros) ===')
print(f"  DTAF1 = {r_null['dtaf1']:.4f}  (expected 0.0)")
print(f"  CBHM  = {r_null['cbhm']:.4f}  (expected 0.0)")

# Missing road only
pred_no_road = np.where(GT == 1, 0, GT)
r_nr = evaluate_all(pred_no_road, GT)
print('\n=== Missing road, correct building ===')
print(f"  DTAF1 = {r_nr['dtaf1']:.4f}")
print(f"  CBHM  = {r_nr['cbhm']:.4f}  (harmonic mean collapses when clDice=0)")

# Missing building only
pred_no_bld = np.where(GT == 2, 0, GT)
r_nb = evaluate_all(pred_no_bld, GT)
print('\n=== Missing building, correct road ===')
print(f"  DTAF1 = {r_nb['dtaf1']:.4f}")
print(f"  CBHM  = {r_nb['cbhm']:.4f}  (harmonic mean collapses when BF=0)")

## 3. Sensitivity Sweeps

In [ ]:
# Helper
def score(pred, gt):
    d = dtaf1(pred, gt, ROAD_CONFIG)
    c = cbhm(pred, gt, linear_classes=[1], polygon_classes=[2])
    ri = iou((pred==1), (gt==1))
    bi = iou((pred==2), (gt==2))
    cl = cldice((pred==1), (gt==1))
    bf = boundary_f1((pred==2), (gt==2), tolerance=2)
    return {
        'dtaf1': d['dtaf1'], 'cbhm': c['cbhm'],
        'road_iou': ri, 'bld_iou': bi,
        'cldice': cl['cldice'], 'bf': bf['bf'],
    }

### 3a. Road offset — IoU vs DTAF1 vs clDice

In [ ]:
offsets = list(range(0, 25, 2))
s_offset = {'offset': [], 'dtaf1': [], 'road_iou': [], 'cldice': []}

for off in offsets:
    pred_road = np.roll(make_road(), off, axis=1)
    pred = np.zeros(SHAPE, dtype=np.uint8)
    pred[pred_road > 0] = 1
    pred[GT == 2] = 2
    s = score(pred, GT)
    s_offset['offset'].append(off)
    s_offset['dtaf1'].append(s['dtaf1'])
    s_offset['road_iou'].append(s['road_iou'])
    s_offset['cldice'].append(s['cldice'])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(s_offset['offset'], s_offset['road_iou'], 'r--o', label='Road IoU')
ax.plot(s_offset['offset'], s_offset['cldice'],   'b-s',  label='clDice')
ax.plot(s_offset['offset'], s_offset['dtaf1'],    'g-^',  label='DTAF1 (d_road=10px)')
ax.axvline(x=10, color='gray', linestyle=':', alpha=0.7, label='tolerance boundary (10px)')
ax.set_xlabel('Road offset (pixels)')
ax.set_ylabel('Score')
ax.set_title('Experiment 1: Road centerline offset sweep')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()
print('Key insight: IoU collapses immediately; clDice/DTAF1 tolerate offsets within the tolerance radius.')

### 3b. Road breakage — random pixel deletion

In [ ]:
fractions = [i/10 for i in range(11)]
rng = np.random.default_rng(42)
road_pixels = np.argwhere(GT == 1)
s_break = {'frac': [], 'dtaf1': [], 'road_iou': [], 'cldice': []}

for frac in fractions:
    pred = GT.copy()
    if frac > 0:
        n_rm = int(frac * len(road_pixels))
        idx = rng.choice(len(road_pixels), n_rm, replace=False)
        for r, c in road_pixels[idx]:
            pred[r, c] = 0
    s = score(pred, GT)
    s_break['frac'].append(frac)
    s_break['dtaf1'].append(s['dtaf1'])
    s_break['road_iou'].append(s['road_iou'])
    s_break['cldice'].append(s['cldice'])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(s_break['frac'], s_break['road_iou'], 'r--o', label='Road IoU')
ax.plot(s_break['frac'], s_break['cldice'],   'b-s',  label='clDice')
ax.plot(s_break['frac'], s_break['dtaf1'],    'g-^',  label='DTAF1')
ax.set_xlabel('Fraction of road pixels removed')
ax.set_ylabel('Score')
ax.set_title('Experiment 2: Road breakage sweep')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

### 3c. Building erosion — shape shrinkage

In [ ]:
radii = list(range(0, 12))
gt_b = (GT == 2).astype(np.uint8)
s_eros = {'radius': [], 'dtaf1': [], 'bld_iou': [], 'bf': []}

for r in radii:
    pred_b = binary_erosion(gt_b, iterations=r).astype(np.uint8) if r > 0 else gt_b
    pred = np.zeros(SHAPE, dtype=np.uint8)
    pred[GT == 1] = 1
    pred[pred_b > 0] = 2
    s = score(pred, GT)
    s_eros['radius'].append(r)
    s_eros['dtaf1'].append(s['dtaf1'])
    s_eros['bld_iou'].append(s['bld_iou'])
    s_eros['bf'].append(s['bf'])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(s_eros['radius'], s_eros['bld_iou'], 'r--o', label='Building IoU')
ax.plot(s_eros['radius'], s_eros['bf'],       'b-s',  label='Boundary F1')
ax.plot(s_eros['radius'], s_eros['dtaf1'],    'g-^',  label='DTAF1')
ax.set_xlabel('Erosion radius (pixels)')
ax.set_ylabel('Score')
ax.set_title('Experiment 3: Building erosion sweep')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

### 3d. Road width — over-thick prediction

In [ ]:
thicknesses = list(range(1, 22, 2))
gt_road = make_road(thickness=3)
gt_map = np.zeros(SHAPE, dtype=np.uint8)
gt_map[gt_road > 0] = 1
gt_map[GT == 2] = 2

s_thick = {'thickness': [], 'dtaf1': [], 'road_iou': [], 'cldice': []}

for t in thicknesses:
    pred_road = make_road(thickness=t)
    pred = np.zeros(SHAPE, dtype=np.uint8)
    pred[pred_road > 0] = 1
    pred[GT == 2] = 2
    s = score(pred, gt_map)
    s_thick['thickness'].append(t)
    s_thick['dtaf1'].append(s['dtaf1'])
    s_thick['road_iou'].append(s['road_iou'])
    s_thick['cldice'].append(s['cldice'])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(s_thick['thickness'], s_thick['road_iou'], 'r--o', label='Road IoU')
ax.plot(s_thick['thickness'], s_thick['cldice'],   'b-s',  label='clDice')
ax.plot(s_thick['thickness'], s_thick['dtaf1'],    'g-^',  label='DTAF1')
ax.axvline(x=3, color='k', linestyle=':', label='GT thickness = 3px')
ax.set_xlabel('Predicted road thickness (pixels)')
ax.set_ylabel('Score')
ax.set_title('Experiment 4: Road width sweep\n(IoU penalises over-thick; clDice/DTAF1 remain high)')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 4. Summary Table

Head-to-head comparison of all metrics on the four degradation scenarios.

In [ ]:
import pandas as pd

def scenario_score(pred):
    s = score(pred, GT)
    return {
        'Road IoU':  round(s['road_iou'], 3),
        'clDice':    round(s['cldice'],   3),
        'Bld IoU':   round(s['bld_iou'],  3),
        'BF Score':  round(s['bf'],       3),
        'DTAF1':     round(s['dtaf1'],    3),
        'CBHM':      round(s['cbhm'],     3),
    }

# Build scenarios
road_offset_5  = make_road(col=69); pred1 = np.zeros(SHAPE, np.uint8); pred1[road_offset_5>0]=1; pred1[GT==2]=2
road_thick_10  = make_road(thickness=10); pred2 = np.zeros(SHAPE, np.uint8); pred2[road_thick_10>0]=1; pred2[GT==2]=2
pred3 = GT.copy()  # 30% road deleted
rp = np.argwhere(GT==1); rng2=np.random.default_rng(0)
for r,c in rp[rng2.choice(len(rp), len(rp)//3, replace=False)]: pred3[r,c]=0
gt_b2 = (GT==2).astype(np.uint8)
pred4 = np.zeros(SHAPE, np.uint8); pred4[GT==1]=1; pred4[binary_erosion(gt_b2, iterations=5)>0]=2

rows = {
    'Perfect (pred=GT)':          scenario_score(GT),
    'Road offset 5px':            scenario_score(pred1),
    'Road over-thick (10px)':     scenario_score(pred2),
    'Road 33% deleted':           scenario_score(pred3),
    'Building eroded 5px':        scenario_score(pred4),
    'Null prediction':            scenario_score(np.zeros_like(GT)),
}

df = pd.DataFrame(rows).T
display(df.style.background_gradient(cmap='RdYlGn', axis=None, vmin=0, vmax=1)
               .set_caption('Metric comparison across degradation scenarios'))

## 5. Key Takeaways

| Scenario | IoU verdict | Actual quality | Correct metric |
|---|---|---|---|
| Road shifted 5px | Low IoU (penalises correct road) | Acceptable | clDice / DTAF1 |
| Road 3× too thick | Low IoU (penalises correct centreline) | Acceptable | clDice / DTAF1 |
| Road broken 33% | Low IoU (correct response) | Poor | All agree |
| Building eroded 5px | Moderate IoU | Moderate | IoU / BF both reasonable |

**Conclusion:** Neither IoU alone nor clDice alone is sufficient for a multiclass map with both roads and buildings.
DTAF1 (unified, single formula, per-class tolerance) and CBHM (harmonic mean of class-optimal metrics) both correctly capture the quality of all feature types simultaneously.